# 📊 Visualisation du Benchmark — Prédiction Usure Outil CNC
**Charge `benchmark_results.json`** produit par `benchmark_tool_wear.py` et génère tous les graphes pertinents.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
import warnings; warnings.filterwarnings('ignore')

# ── Config ──────────────────────────────────────────────────────────────────
RESULTS_PATH = 'benchmark_results.json'   # ← Modifier si besoin

plt.rcParams.update({
    'figure.dpi'       : 120,
    'font.size'        : 10,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})
sns.set_palette('tab10')

# ── Palette modèles ─────────────────────────────────────────────────────────
MODEL_COLORS = [
    '#1565C0','#E65100','#2E7D32','#6A1B9A','#C62828',
    '#00838F','#F57F17','#4E342E','#37474F','#880E4F',
]

print('✅ Imports OK')

In [ ]:
# ── Chargement du JSON ───────────────────────────────────────────────────────
with open(RESULTS_PATH) as f:
    data = json.load(f)

meta        = data['meta']
metrics_raw = data['metrics']
preds_raw   = data['predictions']
TASK        = meta['task']

# Noms courts sans préfixe numérique
def short(name): return name.split('_', 1)[1]

MODEL_NAMES  = [short(k) for k in metrics_raw.keys()]
MODEL_KEYS   = list(metrics_raw.keys())
color_map    = {short(k): MODEL_COLORS[i % len(MODEL_COLORS)]
                for i, k in enumerate(MODEL_KEYS)}

# DataFrame métriques
rows = []
for k, m in metrics_raw.items():
    row = {'model': short(k), **m}
    rows.append(row)
df_m = pd.DataFrame(rows)
df_m = df_m[df_m['status'] == 'ok'].copy()

print(f"Dataset    : {meta['dataset_name']}")
print(f"Tâche      : {TASK.upper()}")
print(f"Cible      : {meta['target_column']}")
print(f"Features   : {meta['features']}")
print(f"Échantill. : {meta['n_samples']}")
print(f"CV Folds   : {meta['cv_folds']}")
print(f"Modèles OK : {len(df_m)}/10")
display(df_m)

---
## 1. 📊 Comparaison des scores (Bar Chart)

In [ ]:
if TASK == 'classification':
    metrics_to_plot = [
        ('f1_macro',          'f1_macro_std' if 'f1_macro_std' in df_m else None, 'F1 Macro (CV mean)'),
        ('accuracy',          'accuracy_std',                                      'Accuracy (CV mean)'),
        ('balanced_accuracy', None,                                                'Balanced Accuracy'),
    ]
else:
    metrics_to_plot = [
        ('r2',   'r2_std',  'R² (CV mean)'),
        ('mae',  None,      'MAE (CV mean)'),
        ('rmse', None,      'RMSE (CV mean)'),
    ]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(6 * len(metrics_to_plot), 6))
if len(metrics_to_plot) == 1: axes = [axes]

for ax, (metric, err_col, label) in zip(axes, metrics_to_plot):
    if metric not in df_m.columns: continue
    df_sorted = df_m.sort_values(metric, ascending=(metric in ['mae', 'rmse']))
    colors    = [color_map.get(m, 'steelblue') for m in df_sorted['model']]
    bars      = ax.barh(df_sorted['model'], df_sorted[metric], color=colors,
                         alpha=0.85, edgecolor='white', linewidth=0.5)

    if err_col and err_col in df_sorted.columns:
        ax.barh(df_sorted['model'], df_sorted[err_col],
                left=df_sorted[metric] - df_sorted[err_col],
                color='black', alpha=0.25, height=0.4)

    for bar, val in zip(bars, df_sorted[metric]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8)

    ax.set_title(label, fontweight='bold')
    ax.set_xlabel(label)
    ax.grid(True, alpha=0.3, axis='x')
    # Meilleur modèle souligné
    best_val = df_sorted[metric].max() if metric not in ['mae', 'rmse'] else df_sorted[metric].min()
    best_idx = (df_sorted[metric] == best_val).values.argmax()
    bars[best_idx].set_edgecolor('gold'); bars[best_idx].set_linewidth(2.5)

plt.suptitle(f'Comparaison des modèles — {meta["dataset_name"]} | {TASK.upper()}',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_scores_comparison.png', bbox_inches='tight', dpi=140)
plt.show()
print('✅ fig1_scores_comparison.png')

---
## 2. ⏱️ Score vs Temps d'entraînement

In [ ]:
main_metric = 'f1_macro' if TASK == 'classification' else 'r2'
metric_label = 'F1 Macro' if TASK == 'classification' else 'R²'

fig, ax = plt.subplots(figsize=(9, 6))

for _, row in df_m.iterrows():
    if main_metric not in row or 'fit_time_s' not in row: continue
    c = color_map.get(row['model'], 'steelblue')
    ax.scatter(row['fit_time_s'], row[main_metric], color=c, s=120, zorder=5,
               edgecolors='white', linewidth=1.2)
    ax.annotate(row['model'], (row['fit_time_s'], row[main_metric]),
                textcoords='offset points', xytext=(6, 4), fontsize=8, color=c)

ax.set_xlabel('Temps CV total (s)', fontweight='bold')
ax.set_ylabel(f'{metric_label} (CV mean)', fontweight='bold')
ax.set_title(f'{metric_label} vs Temps d\'entraînement — {TASK}',
              fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig2_score_vs_time.png', bbox_inches='tight', dpi=140)
plt.show()
print('✅ fig2_score_vs_time.png')

---
## 3. 🎯 Predicted vs Actual (tous les modèles)

In [ ]:
valid_preds = {k: v for k, v in preds_raw.items() if 'y_true' in v and 'y_pred' in v}

n_models  = len(valid_preds)
n_cols    = min(3, n_models)
n_rows    = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, (key, pred) in enumerate(valid_preds.items()):
    ax   = axes_flat[i]
    name = short(key)
    c    = color_map.get(name, 'steelblue')

    y_true = np.array(pred['y_true'])
    y_pred = np.array(pred['y_pred'])

    if TASK == 'regression':
        ax.scatter(y_true, y_pred, alpha=0.5, s=25, color=c,
                   edgecolors='white', linewidth=0.4)
        lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        ax.plot(lims, lims, 'k--', linewidth=1.2, label='Parfait')
        r2  = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        ax.set_title(f'{name}\nR²={r2:.4f} | MAE={mae:.4f}', fontweight='bold', fontsize=9)
        ax.set_xlabel('Réel'); ax.set_ylabel('Prédit')
        ax.text(0.04, 0.92, f'R²={r2:.3f}', transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7), fontsize=9)
    else:
        # Scatter avec jitter pour la classification
        jitter = np.random.uniform(-0.2, 0.2, size=len(y_true))
        try:
            y_t_num = y_true.astype(float) + jitter
            y_p_num = y_pred.astype(float) + jitter
        except:
            unique_vals = sorted(set(y_true) | set(y_pred))
            val2num = {v: j for j, v in enumerate(unique_vals)}
            y_t_num = np.array([val2num[v] for v in y_true], float) + jitter
            y_p_num = np.array([val2num[v] for v in y_pred], float) + jitter
        ax.scatter(y_t_num, y_p_num, alpha=0.4, s=20, color=c)
        acc = np.mean(y_true == y_pred)
        ax.set_title(f'{name}\nAccuracy={acc:.4f}', fontweight='bold', fontsize=9)
        ax.set_xlabel('Réel'); ax.set_ylabel('Prédit')

    ax.grid(True, alpha=0.25)

for j in range(i + 1, len(axes_flat)): axes_flat[j].set_visible(False)

plt.suptitle(f'Prédictions vs Réel — {TASK.upper()} | {meta["dataset_name"]}',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_pred_vs_actual.png', bbox_inches='tight', dpi=140)
plt.show()
print('✅ fig3_pred_vs_actual.png')

---
## 4. 🏆 Radar Chart — profil multi-métriques

In [ ]:
if TASK == 'classification':
    radar_metrics = ['accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted']
    radar_labels  = ['Accuracy', 'Bal. Acc.', 'F1 Macro', 'F1 Weighted']
else:
    # Normalise MAE/RMSE en « score » (1 - val normalisée)
    df_m['mae_score']  = 1 - (df_m['mae']  - df_m['mae'].min())  / (df_m['mae'].max()  - df_m['mae'].min() + 1e-9)
    df_m['rmse_score'] = 1 - (df_m['rmse'] - df_m['rmse'].min()) / (df_m['rmse'].max() - df_m['rmse'].min() + 1e-9)
    df_m['r2_norm']    = (df_m['r2'] - df_m['r2'].min()) / (df_m['r2'].max() - df_m['r2'].min() + 1e-9)
    radar_metrics = ['r2_norm', 'mae_score', 'rmse_score']
    radar_labels  = ['R² (norm)', 'MAE score', 'RMSE score']

N   = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # fermer le polygone

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for _, row in df_m.iterrows():
    vals = [row.get(m, 0) for m in radar_metrics]
    vals += vals[:1]
    c = color_map.get(row['model'], 'steelblue')
    ax.plot(angles, vals, color=c, linewidth=2, label=row['model'])
    ax.fill(angles, vals, color=c, alpha=0.07)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title(f'Radar — Profil multi-métriques ({TASK})', fontweight='bold', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig4_radar.png', bbox_inches='tight', dpi=140)
plt.show()
print('✅ fig4_radar.png')

---
## 5. 🔴 Matrice de confusion (Classification) / Résidus (Régression)

In [ ]:
valid_preds = {k: v for k, v in preds_raw.items() if 'y_true' in v and 'y_pred' in v}

if TASK == 'classification':
    # ── Matrices de confusion ─────────────────────────────────────────────
    n_models = len(valid_preds)
    n_cols   = min(3, n_models)
    n_rows   = (n_models + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for i, (key, pred) in enumerate(valid_preds.items()):
        ax   = axes_flat[i]
        name = short(key)
        y_true, y_pred = np.array(pred['y_true']), np.array(pred['y_pred'])
        cm = confusion_matrix(y_true, y_pred)
        classes = pred.get('classes', sorted(set(y_true)))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=classes, yticklabels=classes,
                    cbar=False, linewidths=0.5)
        acc = np.mean(y_true == y_pred)
        ax.set_title(f'{name}\nAcc={acc:.4f}', fontweight='bold', fontsize=9)
        ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')

    for j in range(i + 1, len(axes_flat)): axes_flat[j].set_visible(False)
    plt.suptitle('Matrices de Confusion — Tous les modèles', fontsize=13, fontweight='bold')

else:
    # ── Graphes de résidus ────────────────────────────────────────────────
    n_models = len(valid_preds)
    n_cols   = min(3, n_models)
    n_rows   = (n_models + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for i, (key, pred) in enumerate(valid_preds.items()):
        ax   = axes_flat[i]
        name = short(key)
        c    = color_map.get(name, 'steelblue')
        y_true = np.array(pred['y_true'], dtype=float)
        y_pred = np.array(pred['y_pred'], dtype=float)
        residuals = y_true - y_pred
        ax.scatter(y_pred, residuals, alpha=0.5, s=25, color=c, edgecolors='white', linewidth=0.3)
        ax.axhline(0, color='black', linestyle='--', linewidth=1.2)
        rmse = np.sqrt(np.mean(residuals ** 2))
        ax.set_title(f'{name}\nRMSE={rmse:.4f}', fontweight='bold', fontsize=9)
        ax.set_xlabel('Valeur prédite'); ax.set_ylabel('Résidu')
        ax.grid(True, alpha=0.25)

    for j in range(i + 1, len(axes_flat)): axes_flat[j].set_visible(False)
    plt.suptitle('Graphes de Résidus — Tous les modèles', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('fig5_confusion_or_residuals.png', bbox_inches='tight', dpi=140)
plt.show()
print('✅ fig5_confusion_or_residuals.png')

---
## 6. 📈 Distribution des erreurs (Boxplot par modèle)

In [ ]:
valid_preds = {k: v for k, v in preds_raw.items() if 'y_true' in v and 'y_pred' in v}

fig, ax = plt.subplots(figsize=(12, 6))

box_data   = []
box_labels = []
box_colors = []

for key, pred in valid_preds.items():
    name   = short(key)
    y_true = np.array(pred['y_true'])
    y_pred = np.array(pred['y_pred'])

    if TASK == 'regression':
        errors = np.abs(y_true.astype(float) - y_pred.astype(float))
    else:
        errors = (y_true != y_pred).astype(float)  # 0 = correct, 1 = erreur

    box_data.append(errors)
    box_labels.append(name)
    box_colors.append(color_map.get(name, 'steelblue'))

bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True,
                medianprops={'color': 'black', 'linewidth': 2})

for patch, c in zip(bp['boxes'], box_colors):
    patch.set_facecolor(c); patch.set_alpha(0.7)

# Ajouter moyennes
for i, d in enumerate(box_data):
    ax.scatter(i + 1, d.mean(), color='black', s=40, zorder=5, marker='D')

ylabel = 'Erreur Absolue' if TASK == 'regression' else 'Taux d\'erreur (0=OK, 1=erreur)'
ax.set_ylabel(ylabel, fontweight='bold')
ax.set_title(f'Distribution des erreurs par modèle — {TASK.upper()}',
              fontweight='bold', fontsize=12)
ax.tick_params(axis='x', rotation=30)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('fig6_error_distribution.png', bbox_inches='tight', dpi=140)
plt.show()
print('✅ fig6_error_distribution.png')

---
## 7. 🥇 Podium — Tableau récapitulatif final

In [ ]:
main_metric = 'f1_macro' if TASK == 'classification' else 'r2'
sort_asc    = False  # R²/F1 : plus grand = meilleur

df_ranked = df_m.sort_values(main_metric, ascending=sort_asc).reset_index(drop=True)
df_ranked.index += 1  # Rang commence à 1
df_ranked.index.name = 'Rang'

medals = {1: '🥇', 2: '🥈', 3: '🥉'}
df_ranked['Médaille'] = df_ranked.index.map(lambda x: medals.get(x, ''))

print(f"\n{'='*70}")
print(f"  CLASSEMENT FINAL — {TASK.upper()} | {meta['dataset_name']}")
print(f"{'='*70}")

if TASK == 'classification':
    cols_show = ['Médaille', 'model', 'accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted', 'fit_time_s']
else:
    cols_show = ['Médaille', 'model', 'r2', 'mae', 'rmse', 'fit_time_s']

cols_show = [c for c in cols_show if c in df_ranked.columns]
display(df_ranked[cols_show].style
    .background_gradient(subset=[main_metric], cmap='RdYlGn',
                         low=0.0, high=1.0 if not sort_asc else None)
    .format({c: '{:.4f}' for c in df_ranked.select_dtypes(float).columns})
    .set_caption(f'Benchmark {TASK} — 5-Fold CV — {meta["timestamp"][:10]}')
)

best = df_ranked.iloc[0]
print(f"\n🏆 Meilleur modèle : {best['model']}")
if TASK == 'classification':
    print(f"   F1 Macro     = {best['f1_macro']:.4f}")
    print(f"   Accuracy     = {best['accuracy']:.4f}")
    print(f"   Bal. Acc.    = {best['balanced_accuracy']:.4f}")
else:
    print(f"   R²           = {best['r2']:.4f}")
    print(f"   MAE          = {best['mae']:.4f}")
    print(f"   RMSE         = {best['rmse']:.4f}")